In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

!pip install -q autoawq transformers accelerate peft
!pip install -q torchao --upgrade

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.3/74.3 kB 3.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 42.4 MB/s eta 0:00:00


In [ ]:
from huggingface_hub import login
login("hf_token")   

In [ ]:
# Imports
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

print(f"CUDA available : {torch.cuda.is_available()}")
print(f"GPU            : {torch.cuda.get_device_name(0)}")

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


CUDA available : True
GPU            : Tesla T4


In [ ]:
# Load base model + LoRA adapter and merge
BASE_MODEL_ID    = "mistralai/Mistral-7B-Instruct-v0.2"
LORA_ADAPTER_ID  = "Shuvam-Maity/edututor-mistral-lora"
MERGED_DIR       = "edututor-mistral-merged"
AWQ_DIR          = "edututor-mistral-awq"
HF_REPO_AWQ      = "Shuvam-Maity/edututor-mistral-awq"

print("Loading base model in float16...")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    torch_dtype=torch.float16,
    device_map="cpu",              
    low_cpu_mem_usage=True
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID)

print("Loading LoRA adapter...")
model = PeftModel.from_pretrained(base_model, LORA_ADAPTER_ID)

print("Merging adapter into base model...")
model = model.merge_and_unload()   

print("Saving merged model locally...")
model.save_pretrained(MERGED_DIR)
tokenizer.save_pretrained(MERGED_DIR)

print(f"✅ Merged model saved to {MERGED_DIR}")

Loading base model in float16...


config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

...

adapter_config.json: 0.00B [00:00, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/13.6M [00:00<?, ?B/s]

Merging adapter into base model...
Saving merged model locally...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Merged model saved to edututor-mistral-merged


In [ ]:
# AWQ Quantization
from awq import AutoAWQForCausalLM

print("Loading merged model for AWQ quantization...")
awq_model = AutoAWQForCausalLM.from_pretrained(
    MERGED_DIR,
    low_cpu_mem_usage=True
)

awq_tokenizer = AutoTokenizer.from_pretrained(MERGED_DIR)

# AWQ config
quant_config = {
    "zero_point": True,
    "q_group_size": 128,
    "w_bit": 4,
    "version": "GEMM"
}

print("Running AWQ quantization — this takes 10-20 minutes...")
awq_model.quantize(
    awq_tokenizer,
    quant_config=quant_config
)

print("Saving AWQ model locally...")
awq_model.save_quantized(AWQ_DIR)
awq_tokenizer.save_pretrained(AWQ_DIR)

print(f"✅ AWQ model saved to {AWQ_DIR}")

/usr/local/lib/python3.12/dist-packages/awq/__init__.py:21: DeprecationWarning: 
I have left this message as the final dev message to help you transition.

Important Notice:
- AutoAWQ is officially deprecated and will no longer be maintained.
- The last tested configuration used Torch 2.6.0 and Transformers 4.51.3.
- If future versions of Transformers break AutoAWQ compatibility, please report the issue to the Transformers project.

Alternative:
- AutoAWQ has been adopted by the vLLM Project: https://github.com/vllm-project/llm-compressor

For further inquiries, feel free to reach out:
- X: https://x.com/casper_hansen_
- LinkedIn: https://www.linkedin.com/in/casper-hansen-804005170/

  warnings.warn(_FINAL_DEV_MESSAGE, category=DeprecationWarning, stacklevel=1)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes

Loading merged model for AWQ quantization...


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Running AWQ quantization — this takes 10-20 minutes...


README.md:   0%|          | 0.00/167 [00:00<?, ?B/s]

...

Generating validation split:   0%|          | 0/214670 [00:00<?, ? examples/s]

AWQ: 100%|██████████| 32/32 [1:00:18<00:00, 113.08s/it]


Saving AWQ model locally...


Writing model shards: 0it [00:00, ?it/s]

✅ AWQ model saved to edututor-mistral-awq


In [ ]:
# Push AWQ model to HF Hub
from huggingface_hub import HfApi

api = HfApi()
api.create_repo(HF_REPO_AWQ, exist_ok=True)
api.upload_folder(
    folder_path=AWQ_DIR,
    repo_id=HF_REPO_AWQ,
    repo_type="model"
)

print(f"✅ Pushed to huggingface.co/{HF_REPO_AWQ}")

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

✅ Pushed to huggingface.co/Shuvam-Maity/edututor-mistral-awq


In [ ]:
# Size comparison
import os

def get_dir_size(path):
    total = 0
    for f in os.listdir(path):
        fp = os.path.join(path, f)
        if os.path.isfile(fp):
            total += os.path.getsize(fp)
    return round(total / 1e9, 2)

merged_size = get_dir_size(MERGED_DIR)
awq_size    = get_dir_size(AWQ_DIR)

print(f"Merged model size : {merged_size} GB")
print(f"AWQ model size    : {awq_size} GB")
print(f"Size reduction    : {round((1 - awq_size/merged_size) * 100)}%")

Merged model size : 14.49 GB
AWQ model size    : 4.15 GB
Size reduction    : 71%
